In [ ]:
import torch
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from torch.utils.data import DataLoader
from tqdm import tqdm
from pathlib import Path
from importlib import import_module
from maw.datasets.landsat import LandsatDataset
from maw.datasets.sentinel import SentinelDataset
from maw.training.metrics import compute_metrics
from maw.models.resnet import build_resnet50
from maw.models.swinv2 import MultimodalSwinV2TRegressor, ModelConfig
from maw.utils.loaders import build_loaders

In [ ]:
# Model Params
N_WORKERS = 8
PREFETCH_FACTOR = 2
PERSISTENT_WORKERS = True

CKPT_STEM = "RESNET50-NOINDIAWEIGHTS-LR0.001-WD0.001-DP0.3"       # set to stem of baseline model checkpoint names
MODEL_CONFIG_PATH = "maw.configs.sentinel_resnet50"               # path to model configs from package root

In [ ]:
# Paths to relevant files
HOME_DIR = Path.home() / "Projects/mapping-asia-wealth"
PREDICTIONS_OUTDIR = HOME_DIR / "data-clean/outputs/predictions/"        # path to save model predictions
EMBEDDINGS_OUTDIR = HOME_DIR / "data-clean/outputs/embeddings/"          # path to save model embeddings

In [ ]:
PREDICTIONS_OUTDIR.mkdir(exist_ok=True, parents=True)
EMBEDDINGS_OUTDIR.mkdir(exist_ok=True, parents=True)

## Experiment Set-up

We aim to predict a household asset index from a 224 x 224 px image patch. Each patch has six spectral bands -- Red, Green, Blue, Near Infrared (NIR), Short-wave Infrared I (SWIR1), and Short-wave Infrared II (SWIR2). For each survey enumeration year, we build a three-year median composite (centered on that year) of all cloud-free Tier 1 images from Landsat 5, 7, 8, and 9 in the USGS/NASA Landsat archive on Google Earth Engine. To reduce the influence of outliers, we clip each band to its empirical 2nd and 98th percentiles (computed across the entire dataset) and then apply the USGS-recommended scaling factors so that reflectance values fall within the standard input range.

Our target variable is either an Absolute Wealth Index defined as the first principal component of asset-ownership survey questions. To maintain consistency, we pool all clusters from DHS5 onward and compute one global PCA using only the subset of asset questions common across those rounds. We define a secondary Relative Wealth Index based on the Demographic and Health Surveys (DHS) Wealth Index, which we average and normalised for each survey independently.  Surveys conducted before DHS 5 are excluded because they contain too few overlapping asset-related questions, which would unduly narrow our index definition.


In [ ]:
mod = import_module(MODEL_CONFIG_PATH)

In [ ]:
# Plot random images from Fold 2
_, loader_vis = build_loaders(train_folds=[1], test_fold=2, cfg=mod.cfg)
loader_vis.dataset.transform = mod.cfg.transforms_val.to("cpu")
loader_vis.dataset.plot_random_16()

## 5-fold Spatial Cross-Validation
We divide the dataset into five geographically independent folds—each a set of non-overlapping observation clusters—stratified by country to maintain balance. Clusters are assigned to folds via a greedy algorithm (i.e. iteratively placing the next largest cluster into the fold with the fewest total observations) to minimize size imbalance and avoid spatial leakage. We then perform 5-fold cross-validation (CV): in each iteration, one fold serves as the held-out (validation) set, and the model is trained on the other four folds. After each round, we generate out-of-fold predictions on the held-out fold.

To evaluate overall performance, we pool all out-of-fold predictions across the five iterations and compute our metrics (e.g., R^2, RMSE, MAE) on this aggregate. This ensures every observation is validated exactly once, yielding an unbiased estimate of generalization across countries.


In [ ]:
# Set up dataset and transforms
transforms = mod.cfg.transforms_val.to(mod.cfg.device)

In [ ]:
if mod.cfg.model_type == 'resnet50':
    model = build_resnet50(hidden_dim=mod.cfg.fc_hidden, dropout=mod.cfg.dropout, frozen_backbone=mod.cfg.freeze_backbone,
                           weights=mod.cfg.initial_weights)
elif mod.cfg.model_type == 'swinv2':
    model_config = ModelConfig(head_dropout=mod.cfg.dropout)
    model = MultimodalSwinV2TRegressor(freeze_backbone=mod.cfg.freeze_backbone, cfg=model_config)
else: 
    raise NotImplementedError("Only 'resnet50' and 'swinv2' are currently supported")

model = model.to(mod.cfg.device)

In [ ]:
model_predictions_list = []
for held_out in range(1, mod.cfg.n_folds + 1):

    # Setup Dataloader
    test_query = f"{mod.cfg.fold_column} == {held_out})"
    train_folds = [1, 2, 3, 4, 5]
    train_folds.remove(held_out)
    _, loader = build_loaders(
            train_folds=train_folds, test_fold=held_out, cfg=mod.cfg, num_workers=N_WORKERS,
            prefetch_factor=PREFETCH_FACTOR, persistent_workers=PERSISTENT_WORKERS, weighted=False
        )

    # Load best-performing checkpoint for held_out
    ckpt = f"{CKPT_STEM}-SPLIT{held_out}-FINAL.pt"
    model.load_state_dict(torch.load(mod.cfg.checkpoint_dir / ckpt, map_location=mod.cfg.device))

    preds_list = []
    ids_list = []
    labels_list = []

    model.eval()
    with torch.no_grad():
        for batch in tqdm(loader, desc=f"Est. Performance on Fold {held_out}", leave=True):
            images = transforms(batch["image"].to(mod.cfg.device))
            if mod.cfg.model_type == "swinv2":
                covariates = batch["covariates"].to(mod.cfg.device)
                preds = model(x=images, g=covariates)
            else:
                preds = model(images)
            preds_list.append(preds.cpu())
            ids_list.append(batch["patch_id"])
            labels_list.append(batch["label"])

    # Save embeddings and predictions
    meta = {
        "model_backbone": "resnet50-cnn",
        "model_backbone_name": ckpt,
        "model_backbone_heldoutset": held_out,
    }

    pred_df = loader.dataset.obs_table.copy()
    pred_df["patch_id_training"] = [id for id_tup in ids_list for id in id_tup]
    pred_df["labels"] = torch.cat(labels_list, dim=0).numpy().flatten()
    pred_df["outputs"] = torch.cat(preds_list, dim=0).numpy().flatten()
    pred_df = pred_df.assign(**meta)
    model_predictions_list.append(pred_df)

    test_mae, test_rmse, test_r2 = compute_metrics(y_true=pred_df['labels'].to_numpy(), y_pred=pred_df['outputs'].to_numpy())
    print({"test_mae": test_mae, "test_rmse": test_rmse, "test_r2": test_r2})


In [ ]:
# Save embeddings and predictions
train_predictions = pd.concat(model_predictions_list).reset_index(drop=True)
train_predictions.to_parquet(PREDICTIONS_OUTDIR / f"{mod.cfg.imagery_type}-test-set-{mod.cfg.model_type}-predictions.parquet")

In [ ]:
# Compute validation metrics for pooled countries
test_mae, test_rmse, test_r2 = compute_metrics(y_true=np.array(train_predictions['labels'].to_numpy()), y_pred=np.array(train_predictions["outputs"].to_numpy()))
print("Test R2:" + str(test_r2))

In [ ]:
# Compute validation metrics for countries individually
country_codes = list(train_predictions["country_code"].unique())
country_metrics_list = []
for cc in country_codes:
    subset_df = train_predictions.query(f"country_code == '{cc}'").reset_index()
    test_mae, test_rmse, test_r2 = compute_metrics(y_true=np.array(subset_df["labels"]), y_pred=np.array(subset_df["outputs"]))
    country_metrics_list.append({"country_code": cc, "mae": test_mae, "rmse": test_rmse, "r2": test_r2})

country_metrics = pd.DataFrame(country_metrics_list)
country_metrics

In [ ]:
# ------ Plot Validation Performance for All-Countries AWI, for All Countries (Pooled) ------

# Built scatterplot for Model Predictions
sns.set_theme(style="whitegrid")
sns.scatterplot(x="labels", y="outputs", hue="rural", data=train_predictions, s=50, color="#045a8d", edgecolor="w", alpha=0.5)
min_val = min(train_predictions["labels"].min(), train_predictions["outputs"].min())
max_val = max(train_predictions["labels"].max(), train_predictions["outputs"].max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle="--", color="red", label="Perfect Prediction")

plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Model Predictions vs. True Wealth Index")
plt.legend(["Rural", "Urban"])

plt.tight_layout()
plt.show()

In [ ]:
# ------ Plot Validation Performance on All-Countries AWI, for All Countries (Separately) ------

# Create a FacetGrid with one facet per country
g = sns.FacetGrid(train_predictions, col="country_code", hue="rural", height=4, aspect=1, col_wrap=4)
g.map_dataframe(sns.scatterplot, x="labels", y="outputs", s=50, alpha=0.5, edgecolor="w")

# Plot diagonal line corresponding to the perfect prediction
for ax in g.axes.flatten():
    xlims = ax.get_xlim()
    ylims = ax.get_ylim()
    min_val = min(xlims[0], ylims[0])
    max_val = max(xlims[1], ylims[1])

    ax.plot([min_val, max_val], [min_val, max_val],
            linestyle="--", color="red", label="Perfect Prediction")

    ax.set_xlim(xlims)
    ax.set_ylim(ylims)

g.add_legend()
g.set_axis_labels("Actual", "Predicted")
g.figure.suptitle("Performance by Country", y=1.05)

plt.tight_layout()
plt.show()


In [ ]:
train_predictions